In [1]:
import os, sys
sys.path.append('../')

# 09 特征工程模块 (core.feature_engineering)

提供特征衍生和转换功能。

**数据说明**: 基于 `hscredit_yyp.xlsx`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.feature_engineering import NumExprDerive

init_setting()

# 加载数据
df = pd.read_excel('hscredit_yyp.xlsx')

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 重命名列为英文
df = df.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
    '极光欺诈分6v1': 'score_fraud',
    '青云24': 'score_qingyun',
    '占信V3': 'score_zhanxin'
})

print(f"样本数: {len(df):,}")
print(f"特征数: {df.shape[1]}")
print(f"坏样本率: {df['target'].mean():.2%}")

样本数: 970
特征数: 19
坏样本率: 16.70%


## 1. 表达式特征衍生

NumExprDerive 接受 `derivings` 参数，是一个 (name, expr) 元组列表

In [3]:
# 定义表达式特征 - 使用 derivings 参数
derivings = [
    ('score_ratio_1', 'score_c3 / (score_coral + 1)'),
    ('score_ratio_2', 'score_fraud / (score_qingyun + 1)'),
    ('score_diff_1', 'score_c3 - score_zhanxin'),
    ('score_sum', 'score_c3 + score_coral + score_fraud'),
    ('score_mean', '(score_c3 + score_coral + score_fraud) / 3'),
]

# 创建特征衍生器
expr_deriver = NumExprDerive(derivings=derivings)
expr_deriver.fit(df)
df_derived = expr_deriver.transform(df)

new_features = [name for name, _ in derivings]
print(f"原始特征数: {df.shape[1]}")
print(f"新增特征数: {len(new_features)}")
print(f"总特征数: {df_derived.shape[1]}")

print("\n衍生特征预览:")
display(df_derived[new_features].describe())

AttributeError: 'super' object has no attribute '__sklearn_tags__'

## 2. 特征效果评估

In [4]:
from hscredit.core.metrics import iv

# 评估新特征的IV
iv_results = []
for feat_name in new_features:
    feat_values = df_derived[feat_name].fillna(df_derived[feat_name].median())
    iv_val = iv(df_derived['target'], feat_values, max_n_bins=10)
    iv_results.append({'特征': feat_name, 'IV': iv_val})

iv_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False)
print("=== 衍生特征IV值 ===")
display(iv_df)

NameError: name 'new_features' is not defined

In [5]:
# 可视化IV结果
plt.figure(figsize=(10, 6))
colors = ['green' if iv > 0.3 else 'orange' if iv > 0.1 else 'red' if iv > 0.02 else 'gray' 
          for iv in iv_df['IV']]
plt.barh(iv_df['特征'], iv_df['IV'], color=colors)
plt.axvline(x=0.02, color='red', linestyle='--', alpha=0.5, label='弱预测(0.02)')
plt.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5, label='中等预测(0.1)')
plt.axvline(x=0.3, color='green', linestyle='--', alpha=0.5, label='强预测(0.3)')
plt.xlabel('IV值')
plt.title('衍生特征IV值排序')
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'iv_df' is not defined

<Figure size 1000x600 with 0 Axes>

## 3. 高级表达式示例

In [6]:
# 更复杂的表达式
advanced_derivings = [
    ('score_c3_log', 'log(score_c3 + 1)'),
    ('score_coral_sqrt', 'sqrt(score_coral)'),
    ('score_c3_squared', 'score_c3 ** 2'),
    ('score_combined', 'where(score_c3 > 600, score_c3 * 0.7, score_c3 * 0.3) + score_coral'),
]

advanced_deriver = NumExprDerive(derivings=advanced_derivings)
advanced_deriver.fit(df)
df_advanced = advanced_deriver.transform(df)

print("高级衍生特征:")
advanced_features = [name for name, _ in advanced_derivings]
display(df_advanced[advanced_features].describe())

AttributeError: 'super' object has no attribute '__sklearn_tags__'